### 1. Análises de qualidade e MultiQC

In [ ]:
fastqc \
    -t 4 \
    -o controle_qualidade/fastqc *.fastq.gz

multiqc \
    controle_qualidade/fastqc \
    -o controle_qualidade/multiqc

[Relatório MultiQC - before-trimming](../results/multiqc_before_trim.html)

### 2. Trimmagem - Remoção de adaptadores e filtragem de bases e reads

In [ ]:
for R1 in /media/lgbio-nas1/kassia.miranda/lcWGS_macauba/*_R1_001.fastq.gz; do
    SAMPLE=$(basename "$R1" _R1_001.fastq.gz)
    R2="/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/${SAMPLE}_R2_001.fastq.gz"
    fastp \
        -i "$R1" \
        -I "$R2" \
        -o "/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/controle_qualidade/trimmagem/${SAMPLE}_R1_001.trim.fq.gz" \
        -O "/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/controle_qualidade/trimmagem/${SAMPLE}_R2_001.trim.fq.gz" \
        --detect_adapter_for_pe \
        -j "${SAMPLE}.fastp.trim.json"
        -q 30 \
        -u 40 \
        -w 24
done

[Relatório MultiQC - pós-trimming](../results/multiqc_post_trim.html)

### 3. Indexando genoma de referência

In [ ]:
bwa-mem2 index genome.fasta

### 4. Alinhado os fastq à referência

In [ ]:
#!/usr/bin/env bash

set -euo pipefail

# ---- configuração do projeto (não dá para obter do FASTQ) ----
GENOME="/media/lgbio-nas1/kassia.miranda/lcWGS_macauba/genome/macauba_genoma.fna"
THREADS=14
SORT_THREADS=4
CN="LGBio"
DT="2026-09-09"
LB_PROJETO="LcWGS_Macauba"
OUTDIR="bam_sorted"

mkdir -p "$OUTDIR"

# checa o índice do bwa-mem2 antes de começar
for EXT in 0123 amb ann bwt.2bit.64 pac; do
    [[ -f "${GENOME}.${EXT}" ]] || { echo "ERRO: índice ausente: ${GENOME}.${EXT} (rode: bwa-mem2 index $GENOME)" >&2; exit 1; }
done

for R1 in *_R1_001.trim.fq.gz; do
    R2="${R1/_R1_001/_R2_001}"
    [[ -f "$R2" ]] || { echo "AVISO: sem R2 para $R1, pulando" >&2; continue; }

    # AACANAMS_11-1_S69_L001
    BASE="${R1%_R1_001.trim.fq.gz}"
    # AACANAMS_11-1  (remove _S69_L001)
    SM="$(sed -E 's/_S[0-9]+_L[0-9]{3}$//' <<< "$BASE")"

    OUT="${OUTDIR}/${BASE}.sort.bam"
    TMP="${OUT}.tmp"

    # pula o que já foi feito
    if [[ -s "$OUT" && -s "${OUT}.bai" ]]; then
        echo "OK: $OUT já existe, pulando" >&2
        continue
    fi

    # primeira linha do FASTQ: @instrumento:run:flowcell:lane:tile:x:y 1:N:0:barcode
    HDR="$(zcat "$R1" | head -n 1 || true)"
    IFS=':' read -r _ _ FC LANE _ _ _ _ _ IDX <<< "$HDR"

    if [[ -z "${FC:-}" || -z "${LANE:-}" ]]; then
        echo "ERRO: cabeçalho inesperado em $R1: $HDR" >&2
        exit 1
    fi

    RG="@RG\tID:${SM}.${FC}.${LANE}\tSM:${SM}\tLB:${LB_PROJETO}\tPU:${FC}.${LANE}.${IDX}\tPL:ILLUMINA\tCN:${CN}\tDT:${DT}"

    echo ">>> $BASE" >&2
    echo "    RG: $RG" >&2

    bwa-mem2 mem \
        -t "$THREADS" \
        -R "$RG" \
        "$GENOME" \
        "$R1" "$R2" \
    | samtools sort \
        -@ "$SORT_THREADS" \
        -T "${OUTDIR}/${BASE}.sorttmp" \
        -o "$TMP" -

    mv "$TMP" "$OUT"
    samtools index -@ "$SORT_THREADS" "$OUT"
done